In [16]:
import dagster as dg
import pandas as pd
import requests
import time
import yaml

import sys
sys.path.append('..')

from datetime import date
from utils.utils import pg_engine
from utils.extract_utils import process_schedule_response, _postgres_upsert
from dagster_dbt import get_asset_key_for_source
from transform_data import transform_assets as la

years = [2009,2008,2007,2006,2005,2004]
payroll_tables=[]
for year in years: 
    payroll_table = pd.read_html(f"https://www.spotrac.com/mlb/aav/_/year/{year}/sort/combined_aav_total")[0]
    payroll_table['season'] = year
    payroll_table['Team'] = payroll_table['Team'].str[:3]
    payroll_table['Team'] = payroll_table['Team'].str.strip()

    payroll_tables.append(payroll_table)
    time.sleep(10)
payroll_tables = pd.concat(payroll_tables).dropna()

import os
os.environ['DB_HOST'] = 'localhost'

engine = pg_engine()

payroll_tables['Total AAV'] = payroll_tables['Total AAV'].replace({'\$':'', ',':''}, regex=True).astype(int)

payroll_tables.to_sql('payroll_tables', engine, schema = "spotrac", if_exists="append",
                #method = _postgres_upsert,
                        index=False)

192

In [1]:
import dagster as dg
import pandas as pd
import requests
import yaml

import sys
sys.path.append('..')

from datetime import date
from utils.utils import pg_engine
import utils.extract_utils  as eu
from dagster_dbt import get_asset_key_for_source
from transform_data import transform_assets as la

#ToDo: Allow for endDate to be determined dynamically
#i.e. just end of calendar year? 
params = {"sportId" :1,
        "gameTypes":"R",
        "gameStatus":"F",
        "startDate":"2004-01-01",
        "endDate":"2009-12-31",
        "hydrate":'probablePitcher,stats'
        }

request_response = eu.send_and_process_schedule_request(params, version = 1, pagination="date_interval")

import os
os.environ['DB_HOST'] = 'localhost'

engine = pg_engine()

request_response.drop_duplicates().to_sql('schedule_endpoint', engine, schema = "mlb_api", if_exists="append",
                method = eu._postgres_upsert,
                        index=False)

/Users/boogleybear/Documents/pgh_hobby_projects/bucco_forecasting/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


NameError: name '_postgres_upsert' is not defined

In [2]:
import os
os.environ['DB_HOST'] = 'localhost'

engine = pg_engine()

request_response.to_sql('schedule_endpoint', engine, schema = "mlb_api", if_exists="append",
                method = eu._postgres_upsert,
                        index=False)

In [18]:
request_response = request_response.drop_duplicates()
request_response[request_response.duplicated('gamepk', keep=False)].sort_values('gamepk')

,gamepk,calendar_event_id,gamedate,gamedt,gametype,game_status,home_win,home_score,away_score,home_id,...,home_pitcher_era_postgame,home_pitcher_innings_pitched_postgame,away_pitcher_innings_pitched_postgame,away_pitcher_earned_runs_game,away_pitcher_earned_runs_postgame,home_pitcher_earned_runs_postgame,away_pitcher_innings_pitched_game,home_pitcher_innings_pitched_game,home_pitcher_earned_runs_game,last_updated_datetime
612,20635,14-20635-2004-05-21,2004-07-02,2004-05-21 07:33:00,R,DL,NaN,NaN,NaN,134,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-04-03 21:03:20
16,20635,14-20635-2004-07-02,2004-07-02,2004-07-02 21:10:00,R,F,True,13.0,2.0,134,...,6.35,78.0,22.2,4.0,10.0,55.0,3.0,6.1,2.0,2025-04-03 21:03:34


In [9]:
request_response.drop('calendar_event_id', axis = 1, inplace = True)

In [7]:
response_re

'Freddy Peralta'

In [ ]:
def _extract_probable_pitcher_info(game_response):
    """
    Extracts season stats of probable pitchers listed in game from schedule endpoint response

    Args:
        game_response (dict): Individual game level dict housed within schedule endpoint response
    
    Returns: 
        dict: dict containing home, away probable pitcher season stats for game
    """
    #Find dict containing season stats
    probable_pitchers = {}
    for team, info in game_response['teams'].items():
        probable_pitchers.update({f'{team}_pitcher_' + key: value for key, value in
                                info['probablePitcher']['stats'][3]['stats'].items()})
    probable_pitchers['gamePk'] = game_response['gamePk']

    probable_pitchers = pd.DataFrame.from_dict(probable_pitchers)

    return probable_pitchers

In [ ]:
response.json()['dates'][100]['games'][0]['gamePk']

In [ ]:
for team, info in response.json()['dates'][100]['games'][0]['teams'].items():
    probable_pitcher_dict = info['probablePitcher']['stats'][3]['stats']
    {f'{team}_pitcher_' + key: value for key, value in probable_pitcher_dict.items()}

#probable_pitcher_dict


    #home_pitcher_dict ['gamepk'] = response_game['gamePk']
    #away_pitcher_dict ['gamepk'] = response_game['gamePk']

In [ ]:
response.json()['dates'][100]['games'][0]['teams'].items()

In [15]:
start_date.strftime("%Y-%m-%d")

NameError: name 'start_date' is not defined